# 01 — Schema & Semantic Profiling

**Objetivo:** perfilar columnas física y semánticamente, con una primera clasificación de riesgo temporal/leakage.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Preguntas
- ¿Qué columnas existen realmente en los CSV locales?
- ¿Qué tipos infiere DuckDB?
- ¿Cuál es el missingness y cardinalidad aproximada?
- ¿Qué columnas parecen IDs, fechas, dinero, texto, geografía o PII?
- ¿Qué campos parecen mutables o posteriores a `t0` y requieren auditoría?

Las etiquetas de leakage de este notebook son **hints**, no decisiones finales.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

{'processes': 'SECOP_II_-_Procesos_de_Contratación_20260921.csv',
 'contracts': 'SECOP_II_-_Contratos_Electrónicos_20260921.csv',
 'bidders': 'Proponentes_por_Proceso_SECOP_II_20260921.csv',
 'suppliers': 'SECOP_II_-_Proveedores_Registrados_20260921.csv',
 'additions': 'SECOP_II_-_Adiciones_20260921.csv',
 'sanctions': 'SECOPII_-_Multas_y_Sanciones_20260921.csv',
 'payments': 'SECOP_II_-_Plan_de_pagos_20260921.csv'}

In [2]:
from argos.profiling.schema import profile_all_datasets
from argos.common.reports import write_report

cfg = PROJECT_CONFIG["profiling"]
column_profile = profile_all_datasets(
    FILES,
    sample_size=cfg["csv_sample_size"],
    example_rows=cfg["example_rows"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    column_profile,
    paths.reports / "profiling" / "column_profile.parquet",
)

column_profile.head(30)

,dataset,column,normalized_column,physical_type,semantic_type,row_count,null_count,null_rate,approx_distinct_count,min_value,max_value,example_values,temporal_role_hint,leakage_risk_hint,parse_mode
0,processes,Entidad,entidad,VARCHAR,category_or_text,9204423,0,0.000000e+00,9192,None,None,"[DEPARTAMENTO ADMINISTRATIVO NACIONAL DE ESTADISTICA (DANE), DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILL...",unknown_until_temporal_audit,review,all_varchar_bronze
1,processes,Nit Entidad,nit_entidad,VARCHAR,identifier,9204423,0,0.000000e+00,11701,None,None,"[899999027, 890102018, 900143276]",unknown_until_temporal_audit,review,all_varchar_bronze
2,processes,Departamento Entidad,departamento_entidad,VARCHAR,geography,9204423,0,0.000000e+00,32,None,None,"[Distrito Capital de Bogotá, Atlántico, Putumayo]",unknown_until_temporal_audit,review,all_varchar_bronze
3,processes,Ciudad Entidad,ciudad_entidad,VARCHAR,geography,9204423,0,0.000000e+00,842,None,None,"[Bogotá, Barranquilla, No Definido]",unknown_until_temporal_audit,review,all_varchar_bronze
4,processes,OrdenEntidad,ordenentidad,VARCHAR,category_or_text,9204423,0,0.000000e+00,4,None,None,"[Nacional, Territorial, Corporación Autónoma]",unknown_until_temporal_audit,review,all_varchar_bronze
5,processes,Entidad Centralizada,entidad_centralizada,VARCHAR,category_or_text,9204423,0,0.000000e+00,2,None,None,"[Centralizada, Descentralizada]",unknown_until_temporal_audit,review,all_varchar_bronze
6,processes,ID del Proceso,id_del_proceso,VARCHAR,identifier,9204423,0,0.000000e+00,9907932,None,None,"[CO1.REQ.2577563, CO1.REQ.2085169, CO1.REQ.7128715]",unknown_until_temporal_audit,review,all_varchar_bronze
7,processes,Referencia del Proceso,referencia_del_proceso,VARCHAR,identifier,9204423,0,0.000000e+00,7216308,None,None,"[EDP-545-2022, CD-55-2021-3516, RE-CGBM-049-2024]",unknown_until_temporal_audit,review,all_varchar_bronze
8,processes,PCI,pci,VARCHAR,identifier,9204423,0,0.000000e+00,10233,None,None,"[700474109, 702442096, 703916536]",unknown_until_temporal_audit,review,all_varchar_bronze
9,processes,ID del Portafolio,id_del_portafolio,VARCHAR,identifier,9204423,0,0.000000e+00,9189956,None,None,"[CO1.BDOS.2503732, CO1.BDOS.2027130, CO1.BDOS.6995772]",unknown_until_temporal_audit,review,all_varchar_bronze


In [3]:
# Prioridad de revisión temporal: campos que NO deben entrar automáticamente a un modelo.
column_profile.query("leakage_risk_hint == 'high'")[[
    "dataset", "column", "semantic_type", "temporal_role_hint", "leakage_risk_hint"
]].sort_values(["dataset", "column"])

,dataset,column,semantic_type,temporal_role_hint,leakage_risk_hint
104,contracts,Dias adicionados,category_or_text,likely_post_t0_or_mutable,high
130,contracts,El contrato puede ser prorrogado,category_or_text,likely_post_t0_or_mutable,high
71,contracts,Estado Contrato,category_or_text,mutable_snapshot_state,high
124,contracts,Fecha Fin Liquidacion,date,likely_post_t0_or_mutable,high
123,contracts,Fecha Inicio Liquidacion,date,likely_post_t0_or_mutable,high
131,contracts,Fecha de notificación de prorrogación,date,likely_post_t0_or_mutable,high
87,contracts,Liquidación,category_or_text,likely_post_t0_or_mutable,high
120,contracts,Ultima Actualizacion,category_or_text,likely_post_t0_or_mutable,high
98,contracts,Valor Amortizado,money,likely_post_t0_or_mutable,high
95,contracts,Valor Facturado,money,likely_post_t0_or_mutable,high


In [4]:
# Missingness alto: no implica eliminar. Primero hay que entender si depende de tiempo/segmento.
column_profile.loc[column_profile["null_rate"].ge(0.50), [
    "dataset", "column", "semantic_type", "null_rate", "approx_distinct_count"
]].sort_values("null_rate", ascending=False).head(100)

,dataset,column,semantic_type,null_rate,approx_distinct_count
16,processes,Fecha de Publicacion (Fase Seleccion Precalificacion),date,0.999999,12
15,processes,Fecha de Publicacion (Fase Planeacion Precalificacion),date,0.999999,15
17,processes,Fecha de Publicacion (Manifestacion de Interes),date,0.992504,3201
18,processes,Fecha de Publicacion (Fase Borrador),date,0.981693,3636
46,processes,Fecha Adjudicacion,date,0.908046,4034
131,contracts,Fecha de notificación de prorrogación,date,0.904212,6082
124,contracts,Fecha Fin Liquidacion,date,0.890858,7332
123,contracts,Fecha Inicio Liquidacion,date,0.890851,6397
26,processes,Fecha de Apertura de Respuesta,date,0.875252,3181
27,processes,Fecha de Apertura Efectiva,date,0.853545,4012


## Criterio de salida
Debe existir un `column_profile.parquet` y una lista explícita de campos que requieran revisión temporal antes de cualquier feature engineering.